# Standalone Amenity Scorer Pipeline (End-to-End)
This notebook contains the complete amenity scoring engine without any local imports.
Everything from fetching OpenStreetMap data to calculating the final mathematical index is included here.

In [1]:
import os
import sys
import json
import math
import time
import random
import logging
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
from typing import List, Dict, Any, Tuple, Optional, Union
from concurrent.futures import ThreadPoolExecutor, as_completed


## config.py

In [2]:
"""
COMPREHENSIVE OSM CONFIGURATION FOR INDIA
==========================================

Complete POI tag configuration based on exhaustive OSM India analysis.
Includes ALL amenity, office, railway, building, shop, and leisure tags.

Version: 2.1 (OSM-only, no Google API)
Date: 2026-02-18
Source: OSM Tag Analysis (Bangalore + India-wide verification)
"""

import os

VERSION = "2.1"

# API Configuration (OSM / Overpass only)
OSM_OVERPASS_URL = "https://overpass-api.de/api/interpreter"
API_TIMEOUT = 90
API_MAX_RETRIES = 5
REQUESTS_PER_SECOND = 0.2

# Data source — OSM only (Google API removed)
DATA_SOURCE_MODE = 'osm'

# Cache and log directories — stored at project root (not inside amenity_scorer/)
# Using __file__ makes paths correct regardless of the working directory.
import pathlib as _pl
_ROOT = _pl.Path(__file__).resolve().parent.parent
CACHE_DIR = str(_ROOT / "cache")
LOG_DIR   = str(_ROOT / "logs")
CACHE_TTL_DAYS = 30

# Analysis Radii (meters) — used by FeatureExtractor for multi-radius counts
RADII = [500, 1000, 2000]

# ============================================================================
# COMPREHENSIVE POI QUALITY WEIGHTS
# ============================================================================
# Based on exhaustive OSM India tag analysis
# Includes: amenity=*, office=*, railway=*, building=*, shop=*, leisure=*
# All tags verified to exist in Indian OSM data
# ============================================================================

POI_WEIGHTS = {
    # ========================================================================
    # HEALTHCARE CATEGORY
    # ========================================================================
    'healthcare': {
        # Amenity tags
        'hospital': 3.0,                 # Major hospitals
        'clinic': 2.0,                   # Clinics, dispensaries
        'pharmacy': 1.8,                 # Pharmacies, medical stores
        'doctors': 2.0,                  # Doctor's offices
        'dentist': 1.5,                  # Dental clinics
        'health_centre': 2.0,            # Health centers
        'nursing_home': 1.8,             # Nursing homes
        'veterinary': 1.3,               # Veterinary clinics
        'medical': 2.0,                  # Medical facilities
        'chemist': 1.8,                  # Chemists (India-specific)
        'physiotherapist': 1.5,          # Physiotherapy centers
        'optician': 1.2,                 # Optical stores
        'laboratory': 1.5,               # Medical labs
        'blood_bank': 2.2,               # Blood banks
        'ayurvedic': 1.7,                # Ayurvedic centers (India)
        'homeopathy': 1.6,               # Homeopathy clinics (India)
        'unani': 1.6,                    # Unani medicine (India)
        # Building tags
        'building_hospital': 3.0,        # Hospital buildings
    },
    
    # ========================================================================
    # EDUCATION CATEGORY
    # ========================================================================
    'education': {
        # Amenity tags
        'university': 3.0,               # Universities
        'college': 2.8,                  # Colleges
        'school': 2.5,                   # Schools
        'kindergarten': 2.0,             # Kindergartens, preschools
        'coaching': 1.8,                 # Coaching centers (India)
        'training': 1.8,                 # Training institutes
        'language_school': 1.5,          # Language schools
        'library': 2.3,                  # Libraries
        'music_school': 2.0,             # Music schools
        'driving_school': 1.3,           # Driving schools
        'research_institute': 2.5,       # Research institutes
        'prep_school': 2.3,              # Preparatory schools
        # Building tags
        'building_school': 2.5,          # School buildings
        'building_college': 2.8,         # College buildings
        'building_university': 3.0,      # University buildings
    },
    
    # ========================================================================
    # FINANCE CATEGORY
    # ========================================================================
    'finance': {
        # Amenity tags
        'bank': 2.3,                     # Banks
        'atm': 1.0,                      # ATMs
        'bureau_de_change': 1.5,         # Currency exchange
        'money_transfer': 1.5,           # Money transfer services
        'post_office': 2.0,              # Post offices
        'post_box': 0.8,                 # Post boxes
        'insurance': 1.7,                # Insurance offices
        'financial_advice': 1.8,         # Financial advisors
        'accountant': 1.5,               # Accountants
        'tax_advisor': 1.6,              # Tax consultants
        # Office tags
        'office_insurance': 1.7,         # Insurance offices
        'office_financial': 1.9,         # Financial services
        'office_accountant': 1.6,        # Accounting firms
        # Building tags
        'building_bank': 2.5,            # Bank buildings
    },
    
    # ========================================================================
    # SHOPPING CATEGORY
    # ========================================================================
    'shopping': {
        # Major retail (amenity tags)
        'mall': 3.0,                     # Shopping malls
        'supermarket': 2.5,              # Supermarkets
        'marketplace': 2.0,              # Markets, bazaars
        'department_store': 2.8,         # Department stores
        'convenience': 1.8,              # Convenience stores
        'wholesale': 1.8,                # Wholesale markets (kept higher weight)
        'variety_store': 1.2,            # Variety stores

        # Shop tags (shop=*) — no duplicates
        'shop': 1.5,                     # Generic shop
        'kirana': 1.8,                   # Kirana stores (India)
        'general': 1.5,                  # General stores
        'butcher': 1.2,                  # Butcher shops
        'bakery': 1.2,                   # Bakeries
        'greengrocer': 1.0,              # Vegetable shops
        'seafood': 1.0,                  # Seafood shops
        'deli': 1.2,                     # Delicatessens
        'confectionery': 1.0,            # Sweet shops
        'beverages': 0.8,                # Beverage stores
        'alcohol': 1.0,                  # Liquor stores
        'tea': 0.8,                      # Tea shops
        'coffee': 0.8,                   # Coffee shops
        'furniture': 1.5,                # Furniture stores
        'electronics': 1.8,              # Electronics shops
        'books': 1.3,                    # Book stores
        'clothes': 1.2,                  # Clothing stores
        'shoes': 1.0,                    # Shoe stores
        'toys': 1.0,                     # Toy stores
        'sports': 1.2,                   # Sports goods
        'jewelry': 1.5,                  # Jewelry stores
        'jewellery': 1.5,                # Jewellery (alternate spelling)
        'mobile_phone': 1.5,             # Mobile phone shops
        'hardware': 1.3,                 # Hardware stores
        'florist': 0.8,                  # Flower shops
        'gift': 0.8,                     # Gift shops
        'stationery': 1.0,               # Stationery stores
        'cosmetics': 1.0,                # Cosmetics shops
        'perfumery': 1.0,                # Perfume shops
        'chemist': 1.2,                  # Chemists (retail)
        'medical_supply': 1.3,           # Medical supplies
        'optician': 1.2,                 # Optical stores
        'doityourself': 1.3,             # DIY stores
        'garden_centre': 1.0,            # Garden centers
        'paint': 1.0,                    # Paint shops
        'carpet': 1.0,                   # Carpet stores
        'curtain': 0.8,                  # Curtain shops
        'interior_decoration': 1.2,      # Interior decoration
        'bed': 1.0,                      # Bed stores
        'kitchen': 1.2,                  # Kitchen stores
        'bathroom_furnishing': 1.0,      # Bathroom furnishing
        'car': 1.8,                      # Car dealerships
        'car_parts': 1.3,                # Auto parts
        'car_repair': 1.3,               # Auto repair shops
        'motorcycle': 1.5,               # Motorcycle shops
        'bicycle': 1.0,                  # Bicycle shops
        'tyres': 1.2,                    # Tyre shops
        'pet': 0.8,                      # Pet stores
        'art': 1.0,                      # Art stores
        'craft': 0.8,                    # Craft stores
        'fabric': 0.8,                   # Fabric shops
        'wool': 0.7,                     # Wool shops
        'newsagent': 0.8,                # News agents
        'lottery': 0.5,                  # Lottery shops
        'ticket': 0.8,                   # Ticket counters
        'travel_agency': 1.3,            # Travel agencies
        'laundry': 1.0,                  # Laundries
        'dry_cleaning': 1.0,             # Dry cleaners
        'trade': 1.0,                    # Trade shops
        'antiques': 1.2,                 # Antique stores
        'baby_goods': 1.0,               # Baby goods
        'beauty': 1.0,                   # Beauty salons
        'hairdresser': 1.0,              # Hairdressers
        'gas': 1.0,                      # Gas shops
        'copyshop': 1.0,                 # Copy shops
        'houseware': 1.0,                # Houseware stores
        'computer': 1.5,                 # Computer shops
        'video_games': 1.2,              # Video game shops
        'music': 1.2,                    # Music shops
        'musical_instrument': 1.5,       # Musical instrument shops
        'photo': 1.2,                    # Photo shops
        'camera': 1.5,                   # Camera shops
        'outdoor': 1.5,                  # Outdoor equipment
        'fishing': 1.2,                  # Fishing shops
        'hunting': 1.2,                  # Hunting shops
        'fashion': 1.2,                  # Fashion stores
        'watches': 1.5,                  # Watch shops
        'chocolate': 1.0,                # Chocolate shops
        'tobacco': 0.8,                  # Tobacco shops
        'e-cigarette': 0.8,              # E-cigarette shops
        'vape': 0.8,                     # Vape shops
        'bag': 1.0,                      # Bag shops
        'lighting': 1.2,                 # Lighting shops

        # Building tags
        'building_retail': 1.4,          # Retail buildings
        'building_kiosk': 1.2,           # Kiosks
    },
    
    # ========================================================================
    # FOOD & DINING CATEGORY
    # ========================================================================
    'food': {
        # Amenity tags
        'restaurant': 1.8,               # Restaurants
        'cafe': 1.3,                     # Cafes
        'fast_food': 0.8,                # Fast food outlets
        'food_court': 1.8,               # Food courts
        'bar': 1.0,                      # Bars
        'pub': 1.0,                      # Pubs
        'biergarten': 1.0,               # Beer gardens
        'ice_cream': 0.7,                # Ice cream parlors
        'tea': 0.8,                      # Tea stalls
        'coffee_shop': 1.2,              # Coffee shops
        'bistro': 1.5,                   # Bistros
        'canteen': 1.0,                  # Canteens
        'pizza': 1.0,                    # Pizza places
        'burger': 0.8,                   # Burger joints
        'chicken': 0.8,                  # Chicken shops
        'sandwich': 0.7,                 # Sandwich shops
        'kebab': 0.8,                    # Kebab shops
        'sushi': 1.3,                    # Sushi restaurants
        'noodle': 1.0,                   # Noodle shops
        'pasta': 1.0,                    # Pasta restaurants
        'seafood': 1.3,                  # Seafood restaurants
        'steak_house': 1.5,              # Steakhouses
        'indian': 1.2,                   # Indian restaurants
        'chinese': 1.2,                  # Chinese restaurants
        'italian': 1.3,                  # Italian restaurants
        'internet_cafe': 1.2,            # Internet cafes
    },
    
    # ========================================================================
    # TRANSPORT CATEGORY
    # ========================================================================
    'transport': {
        # Amenity tags
        'bus_stop': 1.0,                 # Bus stops
        'bus_station': 2.5,              # Bus stations/terminals
        'taxi': 0.8,                     # Taxi stands
        'fuel': 1.8,                     # Petrol pumps
        'parking': 1.2,                  # Parking lots
        'parking_entrance': 1.0,         # Parking entrances
        'parking_space': 0.5,            # Parking spaces
        'bicycle_rental': 1.0,           # Bicycle rentals
        'bicycle_parking': 0.7,          # Bicycle parking
        'motorcycle_parking': 0.7,       # Motorcycle parking
        'car_rental': 1.5,               # Car rentals
        'car_wash': 0.8,                 # Car washes
        'charging_station': 1.5,         # EV charging stations
        'car_sharing': 1.3,              # Car sharing
        'ferry_terminal': 2.0,           # Ferry terminals
        'rest_area': 2.0,                # Highway rest areas
        'services': 2.5,                 # Highway services
        'elevator': 1.5,                 # Public elevators
        
        # Aeroway tags (Raw values)
        'aerodrome': 5.0, 'terminal': 4.0, 'helipad': 3.0, 'heliport': 4.0, 'gate': 2.0,
        
        # Aerialway tags (Raw values)
        'station': 3.0, 'cable_car': 3.0, 'gondola': 3.0, 'chair_lift': 3.0,
        
        # Waterway tags (Raw values)
        'dock': 3.0, 'boatyard': 2.0, 'dam': 2.0,
        
        # Railway tags (railway=*) - Prefixed in poi_fetcher
        'railway_station': 3.0,          # Railway stations
        'railway_subway': 3.0,           # Metro/subway stations
        'railway_subway_entrance': 2.5,  # Metro entrances
        'railway_stop': 1.5,             # Railway stops
        'railway_platform': 1.2,         # Railway platforms
        'railway_halt': 1.8,             # Railway halts
        'railway_tram_stop': 2.0,        # Tram stops
        'railway_light_rail': 3.5,       # Light rail
        'railway_monorail': 3.5,         # Monorail
        
        # Public transport tags (public_transport=*)
        'public_transport_station': 2.5,  # PT stations
        'public_transport_platform': 1.2, # PT platforms
        'public_transport_stop_position': 1.0, # PT stop positions
        'public_transport_ferry_terminal': 3.5, # Ferry terminals
        
        # Building tags
        'building_train_station': 3.0,   # Train station buildings
        'building_transportation': 2.5,  # Transportation buildings
        'building_parking': 1.2,         # Parking buildings
    },
    
    # ========================================================================
    # CULTURAL & RECREATION CATEGORY
    # ========================================================================
    'cultural': {
        # Amenity tags
        'theatre': 2.5,                  # Theatres
        'cinema': 2.0,                   # Cinemas
        'museum': 2.8,                   # Museums
        'library': 2.3,                  # Libraries
        'arts_centre': 2.0,              # Arts centers
        'gallery': 1.8,                  # Art galleries
        'place_of_worship': 1.5,         # Religious places
        'park': 1.8,                     # Parks
        'playground': 1.3,               # Playgrounds
        'community_centre': 1.8,         # Community centers
        'social_centre': 1.5,            # Social centers
        'fountain': 1.0,                 # Fountains
        'monument': 1.5,                 # Monuments
        'viewpoint': 1.3,                # Viewpoints
        'attraction': 2.0,               # Tourist attractions
        'artwork': 1.0,                  # Public art
        'clock': 0.8,                    # Public clocks
        'memorial': 1.2,                 # Memorials
        'wayside_shrine': 0.8,           # Wayside shrines
        'events_venue': 2.3,             # Event venues
        'conference_centre': 2.5,        # Conference centers
        'exhibition_centre': 2.3,        # Exhibition centers
        'studio': 1.3,                   # Studios
        'planetarium': 2.7,              # Planetariums
        'monastery': 1.8,                # Monasteries
        
        # Leisure tags (leisure=*)
        'sports_centre': 2.0,            # Sports centers
        'stadium': 2.5,                  # Stadiums
        'swimming_pool': 2.0,            # Swimming pools
        'fitness_centre': 1.8,           # Fitness centers
        'garden': 1.5,                   # Gardens
        'nature_reserve': 2.0,           # Nature reserves
        'marina': 3.5, 'slipway': 2.0, 'fishing': 1.5, 'pitch': 1.5, 
        'track': 1.5, 
        
        # Natural features (New)
        'beach': 4.0, 'peak': 3.0, 'spring': 2.0, 'cave_entrance': 3.0,
        'wood': 1.0, 'scrub': 0.5, 'water': 2.0,
        
        # Man Made features (New)
        'tower': 2.0, 'lighthouse': 3.5, 'pier': 3.0, 'water_tower': 1.5, 
        'windmill': 2.0,
        
        # Building tags
        'building_temple': 1.5,          # Temples
        'building_church': 1.5,          # Churches
        'building_mosque': 1.5,          # Mosques
        'building_cathedral': 2.0,       # Cathedrals
        'building_chapel': 1.3,          # Chapels
        'building_museum': 2.8,          # Museum buildings
        'building_stadium': 2.5,         # Stadium buildings
        'building_cinema': 2.0,          # Cinema buildings
        'building_grandstand': 2.0,      # Grandstands
    },
    
    # ========================================================================
    # PREMIUM AMENITIES CATEGORY
    # ========================================================================
    'premium': {
        # Amenity tags
        'mall': 3.0,                     # Shopping malls
        'hotel': 2.5,                    # Hotels
        'gym': 1.8,                      # Gyms
        'spa': 2.3,                      # Spas
        'golf_course': 3.0,              # Golf courses
        'resort': 3.0,                   # Resorts
        'fitness_centre': 1.8,           # Fitness centers
        'swimming_pool': 2.0,            # Swimming pools
        'sauna': 1.8,                    # Saunas
        'country_club': 2.8,             # Country clubs
        'sports_centre': 2.0,            # Sports centers
        'stadium': 2.5,                  # Stadiums
        'marina': 2.5,                   # Marinas
        'casino': 2.0,                   # Casinos
        'nightclub': 1.5,                # Nightclubs
        
        # Building tags
        'building_hotel': 2.5,           # Hotel buildings
        'building_hostel': 2.0,          # Hostels
        'building_stadium': 2.5,         # Stadium buildings
    },
    
    # ========================================================================
    # ESSENTIAL SERVICES CATEGORY
    # ========================================================================
    'essential': {
        # Amenity tags
        'hospital': 3.0,                 # Hospitals
        'clinic': 2.0,                   # Clinics
        'pharmacy': 1.8,                 # Pharmacies
        'supermarket': 2.5,              # Supermarkets
        'grocery': 2.3,                  # Grocery stores
        'bank': 2.3,                     # Banks
        'atm': 1.0,                      # ATMs
        'post_office': 2.0,              # Post offices
        'police': 2.8,                   # Police stations
        'fire_station': 2.8,             # Fire stations
        'doctors': 2.0,                  # Doctors
        'dentist': 1.5,                  # Dentists
        'fuel': 2.0,                     # Fuel stations
        'convenience': 1.8,              # Convenience stores
        'toilets': 1.5,                  # Public toilets
        'drinking_water': 1.3,           # Drinking water
        'telephone': 0.9,                # Public phones
        'vending_machine': 0.8,          # Vending machines
        'payment_terminal': 1.0,         # Payment terminals
    },
    
    # ========================================================================
    # EMPLOYMENT & BUSINESS CATEGORY
    # ========================================================================
    'employment': {
        # Amenity tags
        'office': 1.8,                   # Generic offices
        'coworking_space': 2.0,          # Coworking spaces
        'research_institute': 2.5,       # Research institutes
        'industrial': 1.3,               # Industrial areas
        'factory': 1.5,                  # Factories
        'warehouse': 1.2,                # Warehouses
        'craft': 1.0,                    # Craft workshops
        'workshop': 1.2,                 # Workshops
        'research': 2.3,                 # Research facilities
        
        # Office tags (office=*)
        'office_company': 1.8,           # Companies
        'office_it': 2.0,                # IT companies
        'office_coworking': 2.0,         # Coworking offices
        'office_lawyer': 1.8,            # Law firms
        'office_estate_agent': 1.5,      # Real estate
        'office_travel_agent': 1.4,      # Travel agents
        'office_newspaper': 1.7,         # Newspapers
        'office_telecommunication': 1.8, # Telecom companies
        'office_logistics': 1.6,         # Logistics companies
        'office_yes': 1.5,               # Generic offices
        'office_educational_institution': 2.0, # Educational offices
        'office_research': 2.3,          # Research offices
        'office_employment_agency': 1.5, # Employment agencies
        'office_advertising_agency': 1.5,# Advertising agencies
        'office_architect': 1.8, 'office_accountant': 1.8, 'office_consulting': 1.8,
        'office_insurance': 1.8, 'office_financial': 2.0, 'office_government': 2.3,
        'office_ngo': 1.5, 'office_notary': 1.8, 'office_political_party': 1.5,
        
        # Craft tags (Mapped to Employment)
        'carpenter': 1.2, 'plumber': 1.2, 'electrician': 1.2, 'shoemaker': 1.0,
        'tailor': 1.0, 'key_cutter': 1.0, 'photographer': 1.5, 
        'electronics_repair': 1.5,
        
        # Building tags
        'building_office': 1.8,          # Office buildings
        'building_commercial': 1.5,      # Commercial buildings
        'building_retail': 1.4,          # Retail buildings
        'building_industrial': 1.3,      # Industrial buildings
    },
    
    # ========================================================================
    # CIVIC & GOVERNMENT CATEGORY
    # ========================================================================
    'civic': {
        # Amenity tags
        'townhall': 2.8,                 # Town halls
        'courthouse': 2.5,               # Courthouses
        'police': 2.8,                   # Police stations
        'fire_station': 2.8,             # Fire stations
        'post_office': 2.0,              # Post offices
        'embassy': 2.5,                  # Embassies
        'public_building': 2.0,          # Public buildings
        'social_facility': 1.8,          # Social facilities
        'recycling': 1.3,                # Recycling centers
        'community_centre': 1.9,         # Community centers
        
        # Office tags (office=*)
        'office_government': 2.3,        # Government offices
        'office_diplomatic': 2.5,        # Diplomatic offices
        'office_ngo': 1.5,               # NGOs
        'office_association': 1.6,       # Associations
        'office_political_party': 1.7,   # Political parties
        'office_religion': 1.5,          # Religious offices
        'office_foundation': 1.6,        # Foundations
        
        # Building tags
        'building_government': 2.7,      # Government buildings
        'building_public': 2.0,          # Public buildings
        'building_fire_station': 2.8,    # Fire station buildings
        'building_police': 2.8,          # Police station buildings
        'building_community_centre': 1.9,# Community center buildings
    }
}

# ============================================================================
# DENSITY THRESHOLDS (POIs per km²)
# ============================================================================
# "Good" density for a well-served Indian urban area within a 1km radius.
# At this density, a location scores ~70 on the density component.
# Excellent (score=100) is 2.5x these values; fair (score=40) is 0.4x.
#
# Calibrated from Indian OSM data:
#   Metro areas (Mumbai, Delhi, Bengaluru): typically 2x-5x these values
#   Urban fringe (Whitefield, Navi Mumbai):  0.8x-1.5x
#   Rural/semi-urban:                        0.05x-0.3x
DENSITY_THRESHOLDS = {
    'healthcare': 5.0,    # ~4 clinics+pharmacies per km² in urban area
    'education':  3.0,    # ~2-3 schools+coaching per km²
    'finance':    4.0,    # ~3 banks+ATMs per km²
    'shopping':  18.0,    # high count: kirana + retail shops are dense
    'food':      12.0,    # restaurants+cafes very dense in cities
    'premium':    2.0,    # malls+hotels less common
    'transport':  4.0,    # bus stops + fuel + parking
    'cultural':   4.0,    # parks+places of worship
    'essential':  10.0,   # pharmacies+ATMs+fuel+convenience
    'employment': 2.0,    # office density
    'civic':      1.5,    # police+post offices sparse even in cities
}

# ============================================================================
# CATEGORY MAPPINGS
# ============================================================================
# Maps POI types to categories for classification
CATEGORIES = {
    'healthcare': [
        'hospital', 'clinic', 'pharmacy', 'doctors', 'dentist',
        'health_centre', 'nursing_home', 'veterinary', 'medical',
        'chemist', 'physiotherapist', 'optician', 'laboratory',
        'blood_bank', 'ayurvedic', 'homeopathy', 'unani',
        'building_hospital'
    ],
    'education': [
        'university', 'college', 'school', 'kindergarten',
        'coaching', 'training', 'language_school', 'library',
        'music_school', 'driving_school', 'research_institute', 'prep_school',
        'building_school', 'building_college', 'building_university'
    ],
    'finance': [
        'bank', 'atm', 'bureau_de_change', 'money_transfer',
        'post_office', 'post_box', 'insurance', 'financial_advice',
        'accountant', 'tax_advisor',
        'office_insurance', 'office_financial', 'office_accountant',
        'building_bank'
    ],
    'shopping': [
        # Major retail
        'mall', 'supermarket', 'marketplace', 'department_store',
        'convenience', 'wholesale', 'variety_store',
        # All shop types
        'shop', 'kirana', 'general', 'butcher', 'bakery', 'greengrocer',
        'seafood', 'deli', 'confectionery', 'beverages', 'alcohol',
        'tea', 'coffee', 'furniture', 'electronics', 'books', 'clothes',
        'shoes', 'toys', 'sports', 'jewelry', 'jewellery', 'mobile_phone',
        'hardware', 'florist', 'gift', 'stationery', 'cosmetics',
        'perfumery', 'chemist', 'medical_supply', 'optician',
        'doityourself', 'garden_centre', 'paint', 'carpet', 'curtain',
        'interior_decoration', 'bed', 'kitchen', 'bathroom_furnishing',
        'car', 'car_parts', 'car_repair', 'motorcycle', 'bicycle', 'tyres',
        'pet', 'art', 'craft', 'fabric', 'wool', 'newsagent',
        'lottery', 'ticket', 'travel_agency', 'laundry', 'dry_cleaning',
        'trade', 'antiques', 'baby_goods', 'beauty', 'hairdresser',
        'gas', 'copyshop', 'houseware', 'computer', 'video_games',
        'music', 'musical_instrument', 'photo', 'camera', 'outdoor',
        'fishing', 'hunting', 'fashion', 'watches', 'chocolate',
        'tobacco', 'e-cigarette', 'vape', 'bag', 'lighting',
        'building_retail', 'building_kiosk'
    ],
    'food': [
        'restaurant', 'cafe', 'fast_food', 'food_court', 'bar',
        'pub', 'biergarten', 'ice_cream', 'tea', 'coffee_shop',
        'bistro', 'canteen', 'pizza', 'burger', 'chicken',
        'sandwich', 'kebab', 'sushi', 'noodle', 'pasta',
        'seafood', 'steak_house', 'indian', 'chinese', 'italian',
        'internet_cafe'
    ],
    'transport': [
        'bus_stop', 'bus_station', 'taxi', 'fuel', 'parking',
        'parking_entrance', 'parking_space', 'bicycle_rental',
        'bicycle_parking', 'motorcycle_parking', 'car_rental',
        'car_wash', 'charging_station', 'car_sharing',
        'ferry_terminal', 'rest_area', 'services', 'elevator',
        # Aeroway
        'aerodrome', 'terminal', 'helipad', 'heliport', 'gate',
        # Aerialway
        'station', 'cable_car', 'gondola', 'chair_lift',
        # Waterway
        'dock', 'boatyard', 'dam',
        # Railway
        'railway_station', 'railway_subway', 'railway_subway_entrance',
        'railway_stop', 'railway_platform', 'railway_halt',
        'railway_tram_stop', 'railway_light_rail', 'railway_monorail',
        # Public transport
        'public_transport_station', 'public_transport_platform',
        'public_transport_stop_position', 'public_transport_ferry_terminal',
        # Buildings
        'building_train_station', 'building_transportation', 'building_parking'
    ],
    'cultural': [
        'theatre', 'cinema', 'museum', 'library', 'arts_centre',
        'gallery', 'place_of_worship', 'park', 'playground',
        'community_centre', 'social_centre', 'fountain',
        'monument', 'viewpoint', 'attraction', 'artwork',
        'clock', 'memorial', 'wayside_shrine', 'events_venue',
        'conference_centre', 'exhibition_centre', 'studio',
        'planetarium', 'monastery',
        'sports_centre', 'stadium', 'swimming_pool', 'fitness_centre',
        'garden', 'nature_reserve',
        # Leisure extras
        'marina', 'slipway', 'fishing', 'pitch', 'track',
        # Natural features
        'beach', 'peak', 'spring', 'cave_entrance', 'wood', 'scrub', 'water',
        # Man-made landmarks
        'tower', 'lighthouse', 'pier', 'water_tower', 'windmill',
        # Buildings
        'building_temple', 'building_church', 'building_mosque',
        'building_cathedral', 'building_chapel', 'building_museum',
        'building_stadium', 'building_cinema', 'building_grandstand'
    ],
    'premium': [
        'mall', 'hotel', 'gym', 'spa', 'golf_course', 'resort',
        'fitness_centre', 'swimming_pool', 'sauna', 'country_club',
        'sports_centre', 'stadium', 'marina', 'casino', 'nightclub',
        'building_hotel', 'building_hostel', 'building_stadium'
    ],
    'essential': [
        'hospital', 'clinic', 'pharmacy', 'supermarket', 'grocery',
        'bank', 'atm', 'post_office', 'police', 'fire_station',
        'doctors', 'dentist', 'fuel', 'convenience',
        'toilets', 'drinking_water', 'telephone', 'vending_machine',
        'payment_terminal'
    ],
    'employment': [
        'office', 'coworking_space', 'research_institute', 'industrial',
        'factory', 'warehouse', 'craft', 'workshop', 'research',
        'office_company', 'office_it', 'office_coworking', 'office_lawyer',
        'office_estate_agent', 'office_travel_agent', 'office_newspaper',
        'office_telecommunication', 'office_logistics', 'office_yes',
        'office_educational_institution', 'office_research',
        'office_employment_agency', 'office_advertising_agency',
        'office_architect', 'office_accountant', 'office_consulting',
        'office_insurance', 'office_financial', 'office_government',
        'office_ngo', 'office_notary', 'office_political_party',
        # Craft workers
        'carpenter', 'plumber', 'electrician', 'shoemaker',
        'tailor', 'key_cutter', 'photographer', 'electronics_repair',
        # Buildings
        'building_office', 'building_commercial', 'building_retail',
        'building_industrial'
    ],
    'civic': [
        'townhall', 'courthouse', 'police', 'fire_station',
        'post_office', 'embassy', 'public_building', 'social_facility',
        'recycling', 'community_centre',
        'office_government', 'office_diplomatic', 'office_ngo',
        'office_association', 'office_political_party', 'office_religion',
        'office_foundation',
        'building_government', 'building_public', 'building_fire_station',
        'building_police', 'building_community_centre'
    ]
}

# ============================================================================
# CATEGORY WEIGHTS FOR FINAL AMENITY INDEX
# ============================================================================
# Sum = 1.0
CATEGORY_WEIGHTS = {
    'essential': 0.24,
    'healthcare': 0.17,
    'education': 0.14,
    'transport': 0.11,
    'finance': 0.09,
    'shopping': 0.08,
    'food': 0.05,
    'cultural': 0.04,
    'premium': 0.03,
    'employment': 0.03,
    'civic': 0.02
}

# ============================================================================
# COMPONENT WEIGHTS FOR CATEGORY SCORING
# ============================================================================
# Sum = 1.0
COMPONENT_WEIGHTS = {
    'density': 0.25,
    'proximity': 0.20,
    'quality': 0.20,
    'accessibility': 0.15,
    'spatial': 0.10,
    'economic': 0.10
}

# ============================================================================
# ADVANCED SCORING PARAMETERS
# ============================================================================

# Logarithmic Density Scaling
DENSITY_LOG_NORMALIZATION = 3

# Exponential Proximity Decay
PROXIMITY_DECAY_RATE_KM = 1.0
PROXIMITY_DECAY_RATE_AVERAGE_KM = 2.0

# Category-Specific Proximity Decay Rates
# Category-Specific Proximity Decay Rates (lambda in exponential e^{-lambda*d})
# Higher lambda = steeper decay = scoring penalizes distance more aggressively.
# Healthcare/essential: people walk or take auto-rickshaw → sharp decay expected
# Cultural/employment: people drive → softer decay
# At lambda=1.5: 0.5km→47, 1.0km→22, 2.0km→5   (healthcare urgency)
# At lambda=0.8: 0.5km→67, 1.0km→45, 2.0km→20   (employment / driven)
CATEGORY_PROXIMITY_DECAY_RATES = {
    'essential':   1.2,   # pharmacies, ATMs — need nearby (was 0.8)
    'healthcare':  1.5,   # clinics, hospitals — urgent; 2km should score <5
    'education':   1.0,   # schools — moderate walk
    'shopping':    1.0,   # kirana to mall — varies; moderate
    'food':        1.0,   # restaurants — moderate walk (was 0.9)
    'transport':   1.5,   # bus stops — very walkable; far = poor
    'finance':     1.0,   # banks/ATMs — moderate walk
    'cultural':    0.8,   # parks, temples — farther travel acceptable
    'premium':     0.7,   # malls, hotels — driven to; softest decay
    'employment':  0.7,   # offices — commuted to (was 2.0 — too aggressive)
    'civic':       0.9,   # police, post office — moderate walk
}

# Category-Specific Density Log Normalization
CATEGORY_DENSITY_LOG_NORMALIZATION = {
    'essential': 3.0,
    'healthcare': 2.5,
    'education': 2.8,
    'shopping': 4.0,
    'food': 4.5,
    'transport': 2.0,
    'finance': 3.5,
    'cultural': 3.0,
    'premium': 2.5,
    'employment': 3.5,
    'civic': 2.8,
}

# Relative Density Scaling
RELATIVE_DENSITY_SCALE = 500

# Spatial Clustering Parameters
SPATIAL_CLUSTERING_DIVISOR = 3.0

# Premium Brand Lists (India-Specific)
PREMIUM_BRANDS = {
    'food': ['Starbucks', 'KFC', 'McDonald', 'Pizza Hut', 'Domino', 'Burger King', 'Subway', 'Cafe Coffee Day'],
    'shopping': ['Reliance', 'Big Bazaar', 'DMart', 'Westside', 'Lifestyle', 'Pantaloons', 'Shoppers Stop'],
    'healthcare': ['Apollo', 'Fortis', 'Max', 'Manipal', 'Columbia Asia', 'Narayana', 'KIMS'],
    'finance': ['HDFC', 'ICICI', 'Axis', 'Kotak', 'SBI', 'HSBC', 'Citibank'],
    'premium': ['Gold\'s Gym', 'Fitness First', 'Cult.fit', 'Talwalkars'],
}

# Quality Tier Thresholds
QUALITY_THRESHOLDS = {
    'healthcare': {'low': 0.3, 'mid': 0.8, 'high': 1.5},
    'education': {'low': 0.2, 'mid': 0.5, 'high': 1.0},
    'shopping': {'low': 0.1, 'mid': 0.3, 'high': 0.8},
    'transport': {'low': 0.2, 'mid': 0.5, 'high': 1.2},
    'default': {'low': 0.5, 'mid': 1.0, 'high': 2.0}
}

# Gravity Model Log Scaling
GRAVITY_LOG_MIN = 0.1
GRAVITY_LOG_MAX = 100.0



# Dominance Penalty Parameters
DOMINANCE_THRESHOLD = 0.5
DOMINANCE_MULTIPLIER = 2.0

# Gini Coefficient Penalty Parameters
GINI_PENALTY_THRESHOLD = 0.4
GINI_PENALTY_MAX = 0.15

# Simpson's Diversity Boost Parameters
SIMPSON_BOOST_MAX = 0.20

# Data Quality Penalty Thresholds
# Format: (max_poi_count, penalty_fraction)
# Used by amenity_calculator.py to apply additive penalties
DATA_QUALITY_POI_THRESHOLDS = {
    'very_sparse': 5,    # < 5 POIs  → 20% penalty
    'sparse':      20,   # < 20 POIs → 10% penalty
    'moderate':    40,   # < 40 POIs → 5% penalty
    # >= 40 POIs → no penalty
}
DATA_QUALITY_PENALTIES = {
    'very_sparse': 0.20,
    'sparse':      0.10,
    'moderate':    0.05,
    'good':        0.00,
}


# ============================================================================
# COMPLIANCE CHECKS
# ============================================================================
# Ensure weights sum to 1.0 (approx)
if abs(sum(CATEGORY_WEIGHTS.values()) - 1.0) > 0.01:
    raise ValueError(f"CATEGORY_WEIGHTS sum to {sum(CATEGORY_WEIGHTS.values())}, expected 1.0")

if abs(sum(COMPONENT_WEIGHTS.values()) - 1.0) > 0.01:
    raise ValueError(f"COMPONENT_WEIGHTS sum to {sum(COMPONENT_WEIGHTS.values())}, expected 1.0")


# India-calibrated target POI distribution for economic scoring
# These represent the expected share of each category in a balanced urban area
ECONOMIC_TARGET_PCT = {
    'essential':   17,   # Essential services (groceries, pharmacy, police)
    'shopping':    15,   # Retail and shops
    'food':        12,   # Restaurants, cafes, food outlets
    'employment':  12,   # Offices, coworking, industrial
    'transport':   10,   # Transit, parking, fuel
    'healthcare':   8,   # Hospitals, clinics, pharmacies
    'cultural':     8,   # Parks, temples, museums, recreation
    'education':    6,   # Schools, colleges, universities
    'finance':      5,   # Banks, ATMs, post offices
    'premium':      4,   # Hotels, gyms, spas
    'civic':        3,   # Government, civic buildings
}

# Category Minimum Score Constraint
CATEGORY_MIN_SCORE = 0.0

# Spatial Clustering Parameters (DBSCAN)
# EPS = 0.003° ≈ 330m — balanced between the original tight value (0.0015° ≈167m)
# At 1km, POIs in dense cities merge into massive blobs. So we use 0.01
# into one giant cluster; 330m captures real walkable POI hubs without over-merging.
SPATIAL_CLUSTERING_EPS = 0.003          # degrees — kept for reference / legacy
SPATIAL_CLUSTERING_EPS_KM = 0.33       # km-space equivalent used by FeatureExtractor
                                         # (coords converted to km before DBSCAN)
SPATIAL_CLUSTERING_MIN_SAMPLES = 2     # minPts=2
# Multi-radius gradient analysis radii (km) — used by FeatureExtractor._gradient_features
GRADIENT_RADII = [0.5, 1.0, 1.5, 2.0]

# Temporal Accessibility Parameters
TEMPORAL_WEEKEND_DAY = 5
TEMPORAL_EVENING_HOUR = 20
TEMPORAL_MORNING_HOUR = 10


# ============================================================================
# QUALITY CLASSIFICATION
# ============================================================================
# Auto-generated comprehensive lists of Premium and Basic POIs per category.
# Used by CategoryScorer (Quality component) and FeatureExtractor (Ratios).
PREMIUM_POIS = {
    "healthcare": ['building_hospital', 'hospital'],
    "education": ['building_college', 'building_school', 'building_university', 'college', 'research_institute', 'school', 'university'],
    "finance": ['building_bank'],
    "shopping": ['department_store', 'mall', 'supermarket'],
    # Premium food: full-service restaurants, food courts, and fine-dining types.
    # Budget/fast-food types are in BASIC_POIS['food'].
    "food": ['food_court', 'restaurant', 'steak_house', 'sushi', 'bistro', 'seafood'],
    "transport": ['aerodrome', 'building_train_station', 'building_transportation', 'bus_station', 'cable_car', 'chair_lift', 'dock', 'gondola', 'helipad', 'heliport', 'public_transport_ferry_terminal', 'public_transport_station', 'railway_light_rail', 'railway_monorail', 'railway_station', 'railway_subway', 'railway_subway_entrance', 'services', 'station', 'terminal'],
    "cultural": ['beach', 'building_museum', 'building_stadium', 'cave_entrance', 'conference_centre', 'lighthouse', 'marina', 'museum', 'peak', 'pier', 'planetarium', 'stadium', 'theatre'],
    "premium": ['building_hotel', 'building_stadium', 'country_club', 'golf_course', 'hotel', 'mall', 'marina', 'resort', 'stadium'],
    "essential": ['fire_station', 'hospital', 'police', 'supermarket'],
    "employment": ['research_institute'],
    "civic": ['building_fire_station', 'building_government', 'building_police', 'courthouse', 'embassy', 'fire_station', 'office_diplomatic', 'police', 'townhall'],
}

BASIC_POIS = {
    "healthcare": ['ayurvedic', 'blood_bank', 'chemist', 'clinic', 'dentist', 'doctors', 'health_centre', 'homeopathy', 'laboratory', 'medical', 'nursing_home', 'optician', 'pharmacy', 'physiotherapist', 'unani', 'veterinary'],
    "education": ['coaching', 'driving_school', 'kindergarten', 'language_school', 'library', 'music_school', 'prep_school', 'training'],
    "finance": ['accountant', 'atm', 'bank', 'bureau_de_change', 'financial_advice', 'insurance', 'money_transfer', 'office_accountant', 'office_financial', 'office_insurance', 'post_box', 'post_office', 'tax_advisor'],
    "shopping": ['alcohol', 'antiques', 'art', 'baby_goods', 'bag', 'bakery', 'bathroom_furnishing', 'beauty', 'bed', 'beverages', 'bicycle', 'books', 'building_kiosk', 'building_retail', 'butcher', 'camera', 'car', 'car_parts', 'car_repair', 'carpet', 'chemist', 'chocolate', 'clothes', 'coffee', 'computer', 'confectionery', 'convenience', 'copyshop', 'cosmetics', 'craft', 'curtain', 'deli', 'doityourself', 'dry_cleaning', 'e-cigarette', 'electronics', 'fabric', 'fashion', 'fishing', 'florist', 'furniture', 'garden_centre', 'gas', 'general', 'gift', 'greengrocer', 'hairdresser', 'hardware', 'houseware', 'hunting', 'interior_decoration', 'jewellery', 'jewelry', 'kirana', 'kitchen', 'laundry', 'lighting', 'lottery', 'marketplace', 'medical_supply', 'mobile_phone', 'motorcycle', 'music', 'musical_instrument', 'newsagent', 'optician', 'outdoor', 'paint', 'perfumery', 'pet', 'photo', 'seafood', 'shoes', 'shop', 'sports', 'stationery', 'tea', 'ticket', 'tobacco', 'toys', 'trade', 'travel_agency', 'tyres', 'vape', 'variety_store', 'video_games', 'watches', 'wholesale', 'wool'],
    "food": ['bar', 'biergarten', 'bistro', 'burger', 'cafe', 'canteen', 'chicken', 'chinese', 'coffee_shop', 'fast_food', 'food_court', 'ice_cream', 'indian', 'internet_cafe', 'italian', 'kebab', 'noodle', 'pasta', 'pizza', 'pub', 'restaurant', 'sandwich', 'seafood', 'steak_house', 'sushi', 'tea'],
    "transport": ['bicycle_parking', 'bicycle_rental', 'boatyard', 'building_parking', 'bus_stop', 'car_rental', 'car_sharing', 'car_wash', 'charging_station', 'dam', 'elevator', 'ferry_terminal', 'fuel', 'gate', 'motorcycle_parking', 'parking', 'parking_entrance', 'parking_space', 'public_transport_platform', 'public_transport_stop_position', 'railway_halt', 'railway_platform', 'railway_stop', 'railway_tram_stop', 'rest_area', 'taxi'],
    "cultural": ['arts_centre', 'artwork', 'attraction', 'building_cathedral', 'building_chapel', 'building_church', 'building_cinema', 'building_grandstand', 'building_mosque', 'building_temple', 'cinema', 'clock', 'community_centre', 'events_venue', 'exhibition_centre', 'fishing', 'fitness_centre', 'fountain', 'gallery', 'garden', 'library', 'memorial', 'monastery', 'monument', 'nature_reserve', 'park', 'pitch', 'place_of_worship', 'playground', 'scrub', 'slipway', 'social_centre', 'sports_centre', 'spring', 'studio', 'swimming_pool', 'tower', 'track', 'viewpoint', 'water', 'water_tower', 'wayside_shrine', 'windmill', 'wood'],
    "premium": ['building_hostel', 'casino', 'fitness_centre', 'gym', 'nightclub', 'sauna', 'spa', 'sports_centre', 'swimming_pool'],
    "essential": ['atm', 'bank', 'clinic', 'convenience', 'dentist', 'doctors', 'drinking_water', 'fuel', 'grocery', 'payment_terminal', 'pharmacy', 'post_office', 'telephone', 'toilets', 'vending_machine'],
    "employment": ['building_commercial', 'building_industrial', 'building_office', 'building_retail', 'carpenter', 'coworking_space', 'craft', 'electrician', 'electronics_repair', 'factory', 'industrial', 'key_cutter', 'office', 'office_accountant', 'office_advertising_agency', 'office_architect', 'office_company', 'office_consulting', 'office_coworking', 'office_educational_institution', 'office_employment_agency', 'office_estate_agent', 'office_financial', 'office_government', 'office_insurance', 'office_it', 'office_lawyer', 'office_logistics', 'office_newspaper', 'office_ngo', 'office_notary', 'office_political_party', 'office_research', 'office_telecommunication', 'office_travel_agent', 'office_yes', 'photographer', 'plumber', 'research', 'shoemaker', 'tailor', 'warehouse', 'workshop'],
    "civic": ['building_community_centre', 'building_public', 'community_centre', 'office_association', 'office_foundation', 'office_government', 'office_ngo', 'office_political_party', 'office_religion', 'post_office', 'public_building', 'recycling', 'social_facility'],
}



NameError: name '__file__' is not defined

## utils.py

In [ ]:
"""
py — Shared utility functions.
"""

import time
import hashlib
import logging
import numpy as np
from datetime import datetime
from pathlib import Path
from typing import Union


# ---------------------------------------------------------------------------
# Math helpers
# ---------------------------------------------------------------------------

def safe_divide(numerator: float, denominator: float, default: float = 0.0) -> float:
    """Divide two numbers, returning `default` on zero/invalid denominator."""
    if denominator == 0:
        return default
    try:
        result = numerator / denominator
        if not (result == result) or abs(result) == float("inf"):
            return default
        return result
    except (ZeroDivisionError, OverflowError, ValueError):
        return default


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Great-circle distance between two points (Haversine formula).

    Uses arctan2 for better numerical stability near the poles.
    Returns distance in kilometres.
    """
    R = 6371.0
    lat1_r, lon1_r = np.radians(lat1), np.radians(lon1)
    lat2_r, lon2_r = np.radians(lat2), np.radians(lon2)
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def gini_coefficient(values: np.ndarray) -> float:
    """
    Gini coefficient of an array (0 = perfect equality, 1 = maximum inequality).

    Uses the standard sorted-array formula.
    """
    if len(values) < 2 or values.sum() == 0:
        return 0.0
    # Ensure no negative values (Gini undefined for negative inputs)
    values = np.maximum(values, 0)
    sorted_v = np.sort(values)
    n = len(sorted_v)
    idx = np.arange(1, n + 1)
    return float(max(0.0, (2 * np.sum(idx * sorted_v)) / (n * sorted_v.sum()) - (n + 1) / n))


def cache_key(lat: float, lon: float, radius_km: float) -> str:
    """MD5 cache key for a (lat, lon, radius) triple (rounded to ~11 m precision)."""
    key = f"{round(lat, 4)}_{round(lon, 4)}_{radius_km}"
    return hashlib.md5(key.encode()).hexdigest()


# ---------------------------------------------------------------------------
# Rate limiter
# ---------------------------------------------------------------------------

class RateLimiter:
    """Enforces a minimum inter-request interval for API calls."""

    def __init__(self, requests_per_second: float):
        self._interval = 1.0 / requests_per_second
        self._last = 0.0

    def wait(self) -> None:
        elapsed = time.time() - self._last
        if elapsed < self._interval:
            time.sleep(self._interval - elapsed)
        self._last = time.time()


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

def setup_logging(name: str = "amenity_scorer", log_dir: Union[str, Path, None] = None) -> logging.Logger:
    """
    Configure file + console logging.

    Log files are written to `log_dir/amenity_YYYYMMDD.log`.
    Defaults to the project-root logs/ directory (LOG_DIR).
    Returns the named logger.
    """
    if log_dir is None:
        try:
            log_dir = _cfg.LOG_DIR
        except Exception:
            log_dir = "logs"
    log_path = Path(log_dir)
    log_path.mkdir(exist_ok=True)
    log_file = log_path / f"amenity_{datetime.now().strftime('%Y%m%d')}.log"

    fmt = "%(asctime)s | %(name)-20s | %(levelname)-8s | %(message)s"
    logging.basicConfig(
        level=logging.INFO,
        format=fmt,
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger(name)



## poi_fetcher.py

In [ ]:
"""
poi_fetcher.py — Fetch POI data from OpenStreetMap via the Overpass API.

Features:
  - Disk-based JSON cache (configurable TTL)
  - Exponential-backoff retry on transient errors
  - Rate limiting (respects Overpass fair-use policy)
  - Comprehensive OSM tag coverage for Indian cities
"""

import json
import logging
import random
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Optional

import requests


logger = logging.getLogger(__name__)


class POIFetcher:
    """
    Fetch and cache Points of Interest from the Overpass API.

    Usage:
        fetcher = POIFetcher(logger)
        pois = fetcher.fetch(lat=19.076, lon=72.877, max_radius_km=2.0)
    """

    OVERPASS_ENDPOINTS = [
        "https://overpass-api.de/api/interpreter",
        "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
    ]

    def __init__(self, logger_instance: Optional[logging.Logger] = None):
        self.logger = logger_instance or logger
        self.cache_dir = Path(CACHE_DIR)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.cache_ttl = timedelta(days=CACHE_TTL_DAYS)
        self.rate_limiter = RateLimiter(requests_per_second=REQUESTS_PER_SECOND)
        self._endpoint_idx = 0

    def fetch(self, lat: float, lon: float, max_radius_km: float = 2.0, force_refresh: bool = False) -> List[Dict]:
        """
        Return all POIs within `max_radius_km` of (lat, lon).

        Checks disk cache first; falls back to Overpass API on miss.
        Each POI dict contains at minimum: poi_type, lat, lon, distance_km.
        """
        key = cache_key(lat, lon, max_radius_km)
        
        # Check cache if not forcing refresh
        if not force_refresh:
            cached = self._load_cache(key)
            if cached is not None:
                self.logger.debug(f"Cache hit for ({lat:.4f}, {lon:.4f})")
                return cached

        # Fetch from API
        pois = self._fetch_from_api(lat, lon, max_radius_km)

        # Always cache — including empty results for genuinely rural areas.
        # Empty results use a 7-day TTL so they recheck periodically without
        # hammering the API every call. Full results use the standard 30-day TTL.
        self._save_cache(key, pois, ttl_days=7 if not pois else self.cache_ttl.days)

        return pois

    def _load_cache(self, key: str) -> Optional[List[Dict]]:
        path = self.cache_dir / f"{key}.json"
        if not path.exists():
            return None
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            cached_at = datetime.fromisoformat(data["cached_at"])
            # Respect per-entry TTL (empty results use 7 days, normal use 30 days)
            ttl = timedelta(days=data.get("ttl_days", self.cache_ttl.days))
            if datetime.now() - cached_at > ttl:
                path.unlink(missing_ok=True)
                return None
            return data["pois"]
        except Exception:
            return None

    def _save_cache(self, key: str, pois: List[Dict], ttl_days: int = None) -> None:
        path = self.cache_dir / f"{key}.json"
        if ttl_days is None:
            ttl_days = self.cache_ttl.days
        try:
            path.write_text(
                json.dumps({"cached_at": datetime.now().isoformat(),
                            "pois": pois, "ttl_days": ttl_days}),
                encoding="utf-8",
            )
        except Exception as exc:
            self.logger.warning(f"Cache write failed: {exc}")

    def _fetch_from_api(self, lat: float, lon: float, radius_km: float) -> List[Dict]:
        """Query Overpass API with exponential-backoff retry."""
        radius_m = int(radius_km * 1000)
        query = self._build_comprehensive_query(lat, lon, radius_m)

        for attempt in range(API_MAX_RETRIES):
            endpoint = self.OVERPASS_ENDPOINTS[self._endpoint_idx % len(self.OVERPASS_ENDPOINTS)]
            try:
                self.rate_limiter.wait()
                resp = requests.post(
                    endpoint,
                    data={"data": query},
                    timeout=API_TIMEOUT,
                    headers={"User-Agent": "amenity-scorer/2.1 (research)"},
                )
                
                # Check for HTTP 429 specifically — rotate endpoint AND wait
                if resp.status_code == 429:
                    wait = 60 + (attempt * 10) + random.uniform(0, 5)
                    self.logger.warning(
                        f"Rate limited (429) on {endpoint}. "
                        f"Rotating endpoint and waiting {wait:.1f}s... (Attempt {attempt+1})"
                    )
                    self._endpoint_idx += 1  # rotate to next endpoint
                    time.sleep(wait)
                    continue

                resp.raise_for_status()
                data = resp.json()
                
                # Check for Overpass-specific in-body errors
                if "remark" in data:
                    remark = data["remark"]
                    if "timeout" in remark.lower() or "runtime error" in remark.lower():
                        raise requests.exceptions.Timeout(f"Overpass remark: {remark}")
                    self.logger.warning(f"Overpass API remark: {remark}")

                elements = data.get("elements", [])
                pois = self._parse_elements(elements, lat, lon)
                
                if not pois:
                    self.logger.info(f"Fetched 0 POIs at ({lat:.4f}, {lon:.4f}). This may be valid for rural areas.")
                else:
                    self.logger.info(f"Fetched {len(pois)} POIs at ({lat:.4f}, {lon:.4f})")
                
                return pois

            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
                wait = (2 ** attempt) + random.uniform(1, 3)
                self.logger.warning(f"Connection/Timeout error: {exc}. Retry {attempt+1}/{API_MAX_RETRIES} in {wait:.1f}s")
                self._endpoint_idx += 1
                time.sleep(wait)

            except requests.exceptions.HTTPError as exc:
                code = exc.response.status_code
                if code in [500, 502, 503, 504]:
                    wait = (2 ** attempt) * 2 + random.uniform(1, 3)
                    self.logger.warning(f"HTTP {code} (attempt {attempt+1}/{API_MAX_RETRIES}), waiting {wait:.1f}s")
                    time.sleep(wait)
                else:
                    self.logger.error(f"HTTP {code} fatal error: {exc}")
                    break

            except Exception as exc:
                self.logger.error(f"Unexpected error: {exc}", exc_info=True)
                break

        self.logger.error(f"All retries exhausted for ({lat:.4f}, {lon:.4f})")
        return []

    def _build_comprehensive_query(self, lat: float, lon: float, radius_m: float) -> str:
        """
        Build Overpass query with ALL OSM tag types + India-specific tags.
        """
        query = f"""
        [out:json][timeout:120];
        (
          /* Primary Amenities */
          node["amenity"](around:{radius_m},{lat},{lon});
          way["amenity"](around:{radius_m},{lat},{lon});
          
          /* Retail & Commercial */
          node["shop"](around:{radius_m},{lat},{lon});
          way["shop"](around:{radius_m},{lat},{lon});
          node["office"](around:{radius_m},{lat},{lon});
          way["office"](around:{radius_m},{lat},{lon});
          node["craft"](around:{radius_m},{lat},{lon});
          way["craft"](around:{radius_m},{lat},{lon});
          
          /* Specialized & Services */
          node["healthcare"](around:{radius_m},{lat},{lon});
          way["healthcare"](around:{radius_m},{lat},{lon});
          node["leisure"](around:{radius_m},{lat},{lon});
          way["leisure"](around:{radius_m},{lat},{lon});
          node["tourism"](around:{radius_m},{lat},{lon});
          way["tourism"](around:{radius_m},{lat},{lon});
          node["sport"](around:{radius_m},{lat},{lon});
          way["sport"](around:{radius_m},{lat},{lon});
          node["emergency"](around:{radius_m},{lat},{lon});
          way["emergency"](around:{radius_m},{lat},{lon});
          
          /* Transport */
          node["public_transport"](around:{radius_m},{lat},{lon});
          way["public_transport"](around:{radius_m},{lat},{lon});
          node["railway"~"^(station|halt|subway|light_rail|monorail)$"](around:{radius_m},{lat},{lon});
          way["railway"~"^(station|halt|subway|light_rail|monorail)$"](around:{radius_m},{lat},{lon});
          node["aeroway"](around:{radius_m},{lat},{lon});
          way["aeroway"](around:{radius_m},{lat},{lon});
          
          /* India Specifics */
          node["amenity"~"^(coaching|training|place_of_worship|taxi|fuel)$"](around:{radius_m},{lat},{lon});
          way["amenity"~"^(coaching|training|place_of_worship|taxi|fuel)$"](around:{radius_m},{lat},{lon});
          node["shop"~"^(kirana|general|convenience|medical|chemist|beauty|hairdresser)$"](around:{radius_m},{lat},{lon});
          way["shop"~"^(kirana|general|convenience|medical|chemist|beauty|hairdresser)$"](around:{radius_m},{lat},{lon});
          
          /* Buildings (only if tagged with relevant usage) */
          node["building"~"^(commercial|office|retail|hospital|school|college|university|government|public|train_station|hotel|stadium|temple|church|mosque|industrial)$"](around:{radius_m},{lat},{lon});
          way["building"~"^(commercial|office|retail|hospital|school|college|university|government|public|train_station|hotel|stadium|temple|church|mosque|industrial)$"](around:{radius_m},{lat},{lon});
        );
        out center tags;
        """
        return query

    def _parse_elements(self, elements: List[Dict], origin_lat: float, origin_lon: float) -> List[Dict]:
        """Convert raw Overpass elements to normalised POI dicts (deduplicated by ID)."""
        pois = []
        seen_ids = set()

        for el in elements:
            # Deduplicate by OSM ID
            el_id = el.get("id")
            if el_id in seen_ids:
                continue
            seen_ids.add(el_id)

            # Resolve coordinates (nodes have lat/lon; ways/relations have center)
            lat = el.get("lat") or (el.get("center") or {}).get("lat")
            lon = el.get("lon") or (el.get("center") or {}).get("lon")
            if lat is None or lon is None:
                continue

            tags = el.get("tags", {})
            poi_type = self._resolve_poi_type(tags)
            if not poi_type:
                continue

            dist = haversine_km(origin_lat, origin_lon, lat, lon)
            pois.append({
                "id": el_id,
                "poi_type": poi_type,
                "lat": lat,
                "lon": lon,
                "distance_km": dist,
                "name": tags.get("name", ""),
                "brand": tags.get("brand", ""),
                "operator": tags.get("operator", ""),
                "opening_hours": tags.get("opening_hours", ""),
                "tags": tags  # Keep raw tags for brand/feature extraction
            })
        return pois

    def _resolve_poi_type(self, tags: Dict) -> str:
        """
        Extract the most specific POI type from OSM tags.

        Priority: amenity > shop > healthcare > leisure > tourism > sport >
                  craft > emergency > aeroway > aerialway > waterway >
                  natural > man_made > public_transport > highway > office >
                  railway > building

        Tag collision note: 'amenity=station' (bus/train station tagged under
        amenity) is returned as 'amenity_station' to avoid collision with
        'station' (aerialway=station — cable-car station) which carries weight
        3.0 in the transport POI_WEIGHTS.
        """
        # Primary tag keys
        primary_keys = [
            "amenity", "shop", "healthcare", "leisure", "tourism",
            "sport", "craft", "emergency",
            "aeroway", "aerialway", "waterway", "natural", "man_made"
        ]
        for key in primary_keys:
            val = tags.get(key)
            if val:
                # Resolve the 'amenity=station' vs 'aerialway=station' collision.
                # Both produce the raw string 'station'; prefix amenity variant
                # so it doesn't accidentally inherit the aerialway weight (3.0).
                if key == "amenity" and val == "station":
                    return "amenity_station"
                return val

        # Public Transport
        if tags.get("public_transport"):
            return f"public_transport_{tags['public_transport']}"

        # Highway tags (specific amenity-like infrastructure)
        if tags.get("highway") in {"bus_stop", "platform", "rest_area", "services", "elevator"}:
            return tags["highway"]

        if tags.get("office"):
            return f"office_{tags['office']}"

        if tags.get("railway"):
            return f"railway_{tags['railway']}"

        # Building fallback (only specific types)
        if tags.get("building") and tags["building"] != "yes":
            return f"building_{tags['building']}"

        return ""



## feature_extractor.py

In [ ]:
"""
feature_extractor.py — Extract 150–400 features from raw POI data.

Covers:
  - Raw counts, densities, and proximities per POI type and radius
  - Category aggregations (healthcare, education, shopping, …)
  - Quality ratios (hospital/clinic, university/school, …)
  - Gravity model accessibility scores
  - Shannon entropy and category balance
  - Spatial clustering (DBSCAN, Nearest-Neighbour Index, Moran's I)
  - Temporal accessibility (opening hours heuristics)
  - Brand / chain presence
  - Multi-radius density gradients
  - Simpson's diversity and Gini coefficient
  - Proximity decay curves
  - Cross-radius gradients and composite scores
"""

import logging
from collections import Counter
from typing import Dict, List, Optional

import numpy as np
from scipy.stats import entropy

try:
    from sklearn.cluster import DBSCAN
    from scipy.spatial import distance_matrix as scipy_distance_matrix
    _SKLEARN = True
except ImportError:
    _SKLEARN = False
    logging.getLogger(__name__).warning("scikit-learn not installed — spatial clustering disabled")


logger = logging.getLogger(__name__)


class FeatureExtractor:
    """
    Extract all features from a list of POI dicts.

    Each POI dict must have at minimum: poi_type (str), distance_km (float).
    Spatial features additionally require: lat (float), lon (float).
    """

    def __init__(self):
        self.radii = RADII                          # e.g. [500, 1000, 2000]
        self.categories = CATEGORIES                # {category: [poi_type, …]}
        self.poi_weights = POI_WEIGHTS              # {category: {poi_type: weight}}
        self.premium_brands = PREMIUM_BRANDS        # {category: [brand_name, …]}
        self.gradient_radii = GRADIENT_RADII        # e.g. [0.5, 1.0, 1.5, 2.0]

    # ------------------------------------------------------------------
    # Entry point
    # ------------------------------------------------------------------

    def extract_all(self, lat: float, lon: float, pois: List[Dict]) -> Dict:
        """
        Extract all features for a location.

        Args:
            lat, lon: Location coordinates.
            pois:     Raw POI list from POIFetcher.

        Returns:
            Flat feature dict (str → numeric).
        """
        # Sanitise: require poi_type and distance_km
        pois = [p for p in pois if p.get("poi_type") and "distance_km" in p]

        features: Dict = {"latitude": lat, "longitude": lon, "total_pois": len(pois)}

        features.update(self._raw_poi_features(pois))
        features.update(self._category_features(features, pois))
        features.update(self._quality_features(pois))
        features.update(self._gravity_features(pois))
        features.update(self._diversity_features(pois))
        # features.update(self._spatial_features(pois)) # Merged into _advanced_spatial_features
        features.update(self._economic_features(pois))
        features.update(self._cross_radius_features(pois))
        features.update(self._composite_features(features, pois))
        features.update(self._temporal_features(pois))
        features.update(self._brand_features(pois))
        features.update(self._gradient_features(pois))
        features.update(self._advanced_spatial_features(pois))
        features.update(self._advanced_diversity_features(pois))
        features.update(self._proximity_decay_features(pois))

        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _raw_poi_features(self, pois: List[Dict]) -> Dict:
        """Count, density, and proximity for every POI type found."""
        features: Dict = {}
        poi_types = {p["poi_type"] for p in pois}

        for poi_type in poi_types:
            type_pois = [p for p in pois if p["poi_type"] == poi_type]
            distances = [p["distance_km"] for p in type_pois]

            for radius_m in self.radii:
                r_km = radius_m / 1000
                in_r = [p for p in type_pois if p["distance_km"] <= r_km]
                area = np.pi * r_km ** 2
                total_in_r = sum(1 for p in pois if p["distance_km"] <= r_km)

                features[f"{poi_type}_count_{radius_m}m"] = len(in_r)
                features[f"{poi_type}_density_{radius_m}m"] = safe_divide(len(in_r), area)
                features[f"{poi_type}_pct_{radius_m}m"] = safe_divide(len(in_r) * 100, total_in_r)

            features[f"nearest_{poi_type}_km"] = min(distances)
            features[f"avg_dist_{poi_type}_km"] = float(np.mean(distances))

        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _category_features(self, raw: Dict, pois: List[Dict]) -> Dict:
        """Aggregate counts, densities, and diversity per category."""
        features: Dict = {}
        for cat, types in self.categories.items():
            cat_pois = [p for p in pois if p.get("poi_type") in types]
            features[f"{cat}_total_count"] = len(cat_pois)
            features[f"{cat}_total_density"] = sum(
                raw.get(f"{t}_density_1000m", 0) for t in types
            )
            unique = {p["poi_type"] for p in cat_pois}
            features[f"{cat}_diversity"] = safe_divide(len(unique), len(types))
        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _quality_features(self, pois: List[Dict]) -> Dict:
        """Premium-to-basic ratios for key categories."""

        def _count_types(type_list: List[str]) -> int:
            return sum(1 for p in pois if p.get("poi_type") in type_list)

        return {
            "health_quality_ratio":    safe_divide(_count_types(PREMIUM_POIS['healthcare']), _count_types(BASIC_POIS['healthcare']) + 1),
            "education_quality_ratio": safe_divide(_count_types(PREMIUM_POIS['education']), _count_types(BASIC_POIS['education']) + 1),
            "retail_sophistication":   safe_divide(_count_types(PREMIUM_POIS['shopping']), _count_types(BASIC_POIS['shopping']) + 1),
            "transport_quality":       safe_divide(_count_types(PREMIUM_POIS['transport']), _count_types(BASIC_POIS['transport']) + 1),
        }

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _gravity_features(self, pois: List[Dict]) -> Dict:
        """Weighted inverse-square accessibility score per category."""
        features: Dict = {}
        for cat, types in self.categories.items():
            score = 0.0
            for p in pois:
                if p.get("poi_type") in types:
                    w = self.poi_weights.get(cat, {}).get(p["poi_type"], 1.0)
                    d = max(p["distance_km"], 0.1)
                    score += w / (d ** 2)
            features[f"gravity_{cat}"] = score
        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _diversity_features(self, pois: List[Dict]) -> Dict:
        features: Dict = {}
        types = [p["poi_type"] for p in pois if p.get("poi_type")]

        if types:
            counts = np.array(list(Counter(types).values()), dtype=float)
            probs = counts / counts.sum()
            features["amenity_entropy"] = float(entropy(probs))
        else:
            features["amenity_entropy"] = 0.0

        features["unique_amenity_types"] = len(set(types))

        cat_counts = [
            sum(1 for p in pois if p.get("poi_type") in t)
            for t in self.categories.values()
        ]
        total = sum(cat_counts)
        if total > 0:
            cat_probs = np.array(cat_counts, dtype=float) / total
            features["category_balance"] = float(entropy(cat_probs))
        else:
            features["category_balance"] = 0.0

        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _economic_features(self, pois: List[Dict]) -> Dict:
        """Estimate economic activity proxies from POI density within 1km."""
        area_1km = np.pi * 1.0 ** 2  # area of 1km radius circle

        def _density(types, r=1.0):
            # Use the correct area for the given radius, not always area_1km
            area = np.pi * r ** 2
            return safe_divide(
                sum(1 for p in pois if p.get("poi_type") in types and p["distance_km"] <= r),
                area,
            )

        emp_density  = _density(CATEGORIES['employment'])
        cons_density = _density(CATEGORIES['shopping'] + CATEGORIES['food'])
        prem_pct     = safe_divide(
            sum(1 for p in pois if p.get("poi_type") in PREMIUM_POIS['premium']),
            len(pois),
        )

        return {
            "employment_density":   emp_density,
            "consumption_intensity": cons_density,
            "premium_presence":     prem_pct,
            "income_proxy_score":   emp_density * 0.4 + cons_density * 0.3 + prem_pct * 100 * 0.3,
        }

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _cross_radius_features(self, pois: List[Dict]) -> Dict:
        def _density(r_km):
            return safe_divide(
                sum(1 for p in pois if p["distance_km"] <= r_km),
                np.pi * r_km ** 2,
            )

        d500, d1000, d2000 = _density(0.5), _density(1.0), _density(2.0)

        def _grad(outer, inner):
            if inner > 0:
                return safe_divide(outer, inner)
            return 1.0  # Neutral if inner is 0 (prevents spurious sprawl detection)

        return {
            "density_gradient_500_1000":  _grad(d1000, d500),
            "density_gradient_1000_2000": _grad(d2000, d1000),
            "overall_density_gradient":   _grad(d2000, d500),
        }

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _composite_features(self, features: Dict, pois: List[Dict]) -> Dict:
        d500  = safe_divide(sum(1 for p in pois if p["distance_km"] <= 0.5), np.pi * 0.25)
        d2000 = safe_divide(sum(1 for p in pois if p["distance_km"] <= 2.0), np.pi * 4.0)

        essential = ["supermarket", "pharmacy", "clinic", "bank", "atm"]
        available = sum(1 for e in essential if features.get(f"{e}_count_1000m", 0) > 0)

        all_types = {t for types in self.categories.values() for t in types}
        covered = sum(1 for t in all_types if features.get(f"{t}_count_1000m", 0) > 0)

        return {
            "centrality_score":    safe_divide(d500, d2000),
            "self_sufficiency":    safe_divide(available, len(essential)),
            "service_completeness": safe_divide(covered, len(all_types)),
        }

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _temporal_features(self, pois: List[Dict]) -> Dict:
        """Opening-hours heuristics (24/7, weekend, evening).

        IMPORTANT: Only POIs with an *explicit* opening_hours tag are counted
        toward the weekend/evening metrics. POIs with no tag are excluded so
        the metrics reflect actual data rather than inflating to ~100% due to
        missing tags (the majority of OSM POIs have no opening_hours).
        A separate coverage key reports what fraction of POIs have any tag.
        """
        features: Dict = {}

        def _temporal(subset):
            if not subset:
                return {"pct_24_7": 0.0, "weekend_availability": 0.0,
                        "evening_availability": 0.0, "hours_coverage_pct": 0.0}
            tagged = [p for p in subset if p.get("opening_hours")]
            n_tagged = len(tagged)
            coverage = n_tagged / len(subset) * 100
            if n_tagged == 0:
                return {"pct_24_7": 0.0, "weekend_availability": 0.0,
                        "evening_availability": 0.0, "hours_coverage_pct": coverage}
            always = weekend = evening = 0
            for p in tagged:
                oh = p["opening_hours"]
                if oh == "24/7":
                    always += 1
                if "24/7" in oh or any(x in oh for x in ("Mo-Su", "Sa", "Su")):
                    weekend += 1
                if "24/7" in oh or any(x in oh for x in ("20:", "21:", "22:")):
                    evening += 1
            return {
                "pct_24_7":             always  / n_tagged * 100,
                "weekend_availability": weekend / n_tagged * 100,
                "evening_availability": evening / n_tagged * 100,
                "hours_coverage_pct":   coverage,
            }

        global_t = _temporal(pois)
        features.update({f"global_{k}": v for k, v in global_t.items()})

        for cat, types in self.categories.items():
            cat_pois = [p for p in pois if p.get("poi_type") in types]
            if cat_pois:
                t = _temporal(cat_pois)
                features[f"{cat}_pct_24_7"]             = t["pct_24_7"]
                features[f"{cat}_weekend_availability"] = t["weekend_availability"]
                features[f"{cat}_evening_availability"] = t["evening_availability"]
                features[f"{cat}_hours_coverage_pct"]   = t["hours_coverage_pct"]

        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _brand_features(self, pois: List[Dict]) -> Dict:
        features: Dict = {}

        def _brand_stats(subset, brand_list):
            if not subset:
                return {"premium_brand_count": 0, "premium_brand_pct": 0.0,
                        "brand_diversity": 0.0, "chain_presence_pct": 0.0}
            n = len(subset)
            premium = 0
            for p in subset:
                text = " ".join([p.get("name", ""), p.get("brand", ""), p.get("operator", "")]).lower()
                if any(b.lower() in text for b in brand_list):
                    premium += 1
            brands = [p.get("brand") or p.get("operator") for p in subset if p.get("brand") or p.get("operator")]
            return {
                "premium_brand_count": premium,
                "premium_brand_pct":   premium / n * 100,
                "brand_diversity":     safe_divide(len(set(brands)), len(brands)) if brands else 0.0,
                "chain_presence_pct":  len(brands) / n * 100,
            }

        all_brands = [b for bl in self.premium_brands.values() for b in bl]
        g = _brand_stats(pois, all_brands)
        features.update({f"global_{k}": v for k, v in g.items()})

        for cat, types in self.categories.items():
            cat_pois = [p for p in pois if p.get("poi_type") in types]
            if cat_pois:
                b = _brand_stats(cat_pois, self.premium_brands.get(cat, all_brands))
                features[f"{cat}_premium_brand_count"] = b["premium_brand_count"]
                features[f"{cat}_premium_brand_pct"]   = b["premium_brand_pct"]
                features[f"{cat}_brand_diversity"]     = b["brand_diversity"]

        return features

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _gradient_features(self, pois: List[Dict]) -> Dict:
        features: Dict = {}
        for cat, types in self.categories.items():
            densities = []
            for r in self.gradient_radii:
                cnt = sum(1 for p in pois if p.get("poi_type") in types and p.get("distance_km", 999) <= r)
                densities.append(safe_divide(cnt, np.pi * r ** 2))

            if len(densities) < 2:
                features[f"{cat}_density_gradient"] = 0.0
                features[f"{cat}_density_monotonic"] = 0.0
                continue

            grads = [densities[i + 1] - densities[i] for i in range(len(densities) - 1)]
            features[f"{cat}_density_gradient"]  = float(np.mean(grads))
            features[f"{cat}_density_monotonic"] = 1.0 if all(g <= 0 for g in grads) else 0.0
            features[f"{cat}_density_pattern"]   = self._classify_density_pattern(grads)

            mean_d = np.mean(densities)
            if mean_d > 0:
                features[f"{cat}_density_stability_cv"] = max(0.0, 100 * (1 - np.std(densities) / mean_d))
            else:
                features[f"{cat}_density_stability_cv"] = 0.0

        return features

    @staticmethod
    def _classify_density_pattern(grads: List[float]) -> str:
        """Classify the spatial density gradient pattern.

        Gradients are density[r+1] - density[r]. Since density naturally
        decreases outward, negative gradients indicate normal outward decay.
          uniform   — density roughly constant across all radii (<0.5 change)
          core      — density drops steeply at first then flattens (dense core, sparse fringe)
          isolated  — single sharp concentration, then near-zero outward
          sprawl    — density increases or stays flat as radius grows (no clear centre)
        """
        if not grads:
            return "unknown"
        if all(abs(g) < 0.5 for g in grads):
            return "uniform"
        # All gradients strongly negative — density falling consistently outward
        if all(g < -0.5 for g in grads):
            return "core"
        # First gradient is large negative, last flattens — concentrated core then fringe
        if len(grads) >= 2 and grads[0] < -2 and grads[-1] > -0.5:
            return "isolated"
        # Density not decreasing monotonically — no clear focal point
        return "sprawl"

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _advanced_spatial_features(self, pois: List[Dict]) -> Dict:
        base = {
            "global_dbscan_n_clusters":     0,
            "global_dbscan_cluster_score":  0.0,
            "global_nearest_neighbor_index": 0.0,
            "global_hotspot_intensity":     0.0,
            "global_morans_i":              0.0,
            "global_distance_gini":         0.0,
        }
        if not _SKLEARN or len(pois) < 3:
            return base

        coords = np.array([[p["lat"], p["lon"]] for p in pois if "lat" in p and "lon" in p])
        if len(coords) < 3:
            return base

        # Convert degree coords to km for correct Euclidean distance at India's latitudes.
        # At ~20°N: 1° lat ≈ 111.32 km, 1° lon ≈ 111.32 × cos(20°) ≈ 104.6 km.
        mean_lat_r = np.radians(coords[:, 0].mean())
        scale = np.array([111.32, 111.32 * np.cos(mean_lat_r)])
        coords_km = coords * scale

        labels = DBSCAN(
            eps=SPATIAL_CLUSTERING_EPS_KM,   # km-space, not degrees
            min_samples=SPATIAL_CLUSTERING_MIN_SAMPLES,
        ).fit(coords_km).labels_

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        cluster_score = safe_divide(n_clusters * 100, len(coords_km))

        # Nearest-Neighbour Index — computed in km-space
        dm = scipy_distance_matrix(coords_km, coords_km)
        np.fill_diagonal(dm, np.inf)
        avg_nn = float(np.mean(np.min(dm, axis=1)))
        area_km2 = np.pi * 2.0 ** 2  # 2km radius catchment in km²
        expected_nn = 0.5 / np.sqrt(safe_divide(len(coords_km), area_km2))
        nni = safe_divide(avg_nn, expected_nn, default=1.0)

        # Hotspot intensity
        if n_clusters > 0:
            sizes = [list(labels).count(i) for i in set(labels) if i != -1]
            hotspot = safe_divide(max(sizes) * 100, len(coords_km))
        else:
            hotspot = 0.0

        result = {
            "global_dbscan_n_clusters":     n_clusters,
            "global_dbscan_cluster_score":  min(100.0, cluster_score),
            "global_nearest_neighbor_index": nni,
            "global_hotspot_intensity":     hotspot,
            "global_morans_i":              self._morans_i(coords_km),
            "global_distance_gini":         float(gini_coefficient(np.array([p["distance_km"] for p in pois]))),
        }

        # Per-category DBSCAN, hotspot, AND per-category NNI
        for cat, types in self.categories.items():
            cat_pois = [p for p in pois if p.get("poi_type") in types]
            if len(cat_pois) >= 3:
                cat_coords = np.array([[p["lat"], p["lon"]] for p in cat_pois if "lat" in p])
                if len(cat_coords) >= 3:
                    # Scale to km-space for this category
                    cat_coords_km = cat_coords * scale
                    cat_labels = DBSCAN(
                        eps=SPATIAL_CLUSTERING_EPS_KM,   # km-space
                        min_samples=SPATIAL_CLUSTERING_MIN_SAMPLES,
                    ).fit(cat_coords_km).labels_
                    n_c = len(set(cat_labels)) - (1 if -1 in cat_labels else 0)
                    result[f"{cat}_dbscan_n_clusters"]    = n_c
                    result[f"{cat}_dbscan_cluster_score"] = min(100.0, safe_divide(n_c * 100, len(cat_coords_km)))
                    if n_c > 0:
                        s = [list(cat_labels).count(i) for i in set(cat_labels) if i != -1]
                        result[f"{cat}_hotspot_intensity"] = safe_divide(max(s) * 100, len(cat_coords_km))
                    else:
                        result[f"{cat}_hotspot_intensity"] = 0.0
                    # Per-category NNI (km-space)
                    cat_dm = scipy_distance_matrix(cat_coords_km, cat_coords_km)
                    np.fill_diagonal(cat_dm, np.inf)
                    cat_avg_nn = float(np.mean(np.min(cat_dm, axis=1)))
                    cat_area_km2 = np.pi * 2.0 ** 2
                    cat_expected_nn = 0.5 / np.sqrt(safe_divide(len(cat_coords_km), cat_area_km2))
                    result[f"{cat}_nearest_neighbor_index"] = safe_divide(cat_avg_nn, cat_expected_nn, default=1.0)

        return result

    @staticmethod
    def _morans_i(coords: np.ndarray) -> float:
        """
        Moran's I spatial autocorrelation (-1 dispersed, 0 random, +1 clustered).

        Uses inverse-distance row-standardised weights and distance-from-centroid
        as the attribute variable.
        """
        if len(coords) < 4:
            return 0.0
        try:
            dm = scipy_distance_matrix(coords, coords).astype(float)
            # Avoid division by zero on diagonal or duplicate points
            with np.errstate(divide='ignore', invalid='ignore'):
                W = 1.0 / dm
            W[np.isinf(W)] = 0.0
            np.fill_diagonal(W, 0.0)  # No self-influence
            row_sums = W.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1.0
            W /= row_sums

            centroid = coords.mean(axis=0)
            x = np.array([np.linalg.norm(c - centroid) for c in coords])
            n, xbar = len(x), x.mean()
            z = x - xbar
            numerator = float(np.dot(z, W @ z))
            denom = float(np.dot(z, z))
            W_sum = float(W.sum())
            if denom == 0 or W_sum == 0:
                return 0.0
            return float(np.clip((n / W_sum) * (numerator / denom), -1.0, 1.0))
        except Exception:
            return 0.0

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _advanced_diversity_features(self, pois: List[Dict]) -> Dict:
        types = [p["poi_type"] for p in pois if p.get("poi_type")]
        if not types:
            return {"global_simpson_diversity": 0.0, "global_gini_coefficient": 0.0,
                    "global_category_balance_gini": 0.0}

        counts = np.array(list(Counter(types).values()), dtype=float)
        n = counts.sum()
        simpson = float(1 - np.sum((counts / n) ** 2)) * 100

        cat_counts = np.array([
            sum(1 for p in pois if p.get("poi_type") in t)
            for t in self.categories.values()
        ], dtype=float)
        cat_gini = gini_coefficient(cat_counts[cat_counts > 0]) if cat_counts.sum() > 0 else 0.0

        return {
            "global_simpson_diversity":    simpson,
            "global_gini_coefficient":     float(gini_coefficient(counts)),
            "global_category_balance_gini": (1 - cat_gini) * 100,
        }

    # ------------------------------------------------------------------
    # ------------------------------------------------------------------

    def _proximity_decay_features(self, pois: List[Dict]) -> Dict:
        base = {
            "global_distance_p25": 0.0, "global_distance_median": 0.0,
            "global_distance_p75": 0.0, "global_distance_p90": 0.0,
            "global_poi_concentration_500m": 0.0, "global_poi_concentration_1000m": 0.0,
            "global_distance_variance": 0.0, "global_distance_skewness": 0.0,
        }
        if not pois:
            return base

        d = np.array([p["distance_km"] for p in pois if "distance_km" in p])
        if len(d) == 0:
            return base

        std = float(np.std(d))
        skew = float(np.mean(((d - d.mean()) / std) ** 3)) if std > 0 else 0.0

        result = {
            "global_distance_p25":             float(np.percentile(d, 25)),
            "global_distance_median":          float(np.percentile(d, 50)),
            "global_distance_p75":             float(np.percentile(d, 75)),
            "global_distance_p90":             float(np.percentile(d, 90)),
            "global_poi_concentration_500m":   float(np.mean(d <= 0.5) * 100),
            "global_poi_concentration_1000m":  float(np.mean(d <= 1.0) * 100),
            "global_distance_variance":        float(np.var(d)),
            "global_distance_skewness":        skew,
        }

        for cat, types in self.categories.items():
            cat_d = np.array([p["distance_km"] for p in pois
                              if p.get("poi_type") in types and "distance_km" in p])
            if len(cat_d) > 0:
                result[f"{cat}_distance_median"]         = float(np.median(cat_d))
                result[f"{cat}_poi_concentration_500m"]  = float(np.mean(cat_d <= 0.5) * 100)
                result[f"{cat}_distance_variance"]       = float(np.var(cat_d))

        return result



## category_scorer.py

In [ ]:
"""
category_scorer.py — Score each amenity category on 6 components.

Components and weights (configurable in COMPONENT_WEIGHTS):
  1. Density      (25%) — POI count relative to urban benchmarks
  2. Proximity    (20%) — Distance to nearest POIs
  3. Quality      (20%) — Premium-to-basic POI ratio
  4. Accessibility(15%) — Gravity-model weighted access score
  5. Spatial      (10%) — Clustering and distribution
  6. Economic     (10%) — Category share vs India-calibrated target
"""

import logging
from collections import Counter
from typing import Dict, List, Tuple

import numpy as np


logger = logging.getLogger(__name__)

# Categories that are naturally mono-type — skip dominance penalty
_SKIP_DOMINANCE = frozenset({"civic", "finance", "premium"})

# Max fractional penalty at full POI-type dominance (share=100% → ×0.8 reduction)
_DOMINANCE_PENALTY_MULTIPLIER = 0.4


class CategoryScorer:
    """
    Calculate a 0–100 score for each amenity category.

    Usage:
        scorer = CategoryScorer()
        result = scorer.score("healthcare", features, pois)
        # result = {"score": 72.4, "components": {...}}
    """

    def __init__(self):
        self.categories    = CATEGORIES
        self.poi_weights   = POI_WEIGHTS
        self.density_thresholds = DENSITY_THRESHOLDS
        self.component_weights  = COMPONENT_WEIGHTS

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def score(self, category: str, features: Dict, pois: List[Dict]) -> Dict:
        """
        Score a single category.

        Args:
            category: Category name (must be a key in CATEGORIES).
            features: Feature dict from FeatureExtractor.
            pois:     Full POI list (all categories).

        Returns:
            {"score": float, "components": {component: float}}
        """
        if category not in self.categories:
            logger.warning(f"Unknown category: {category}")
            return {"score": 0.0, "components": {}}

        cat_pois = [p for p in pois if p.get("poi_type") in self.categories[category]]

        components = {
            "density":       self._density(cat_pois, category),
            "proximity":     self._proximity(cat_pois, category),
            "quality":       self._quality(cat_pois, category),
            "accessibility": self._accessibility(cat_pois, category),
            "spatial":       self._spatial(cat_pois, category, features),
            "economic":      self._economic(cat_pois, pois, category),
        }

        raw_score = sum(
            components[c] * self.component_weights.get(c, 0)
            for c in components
        )

        # Soft dominance penalty: only for large, diverse categories
        if (
            category not in _SKIP_DOMINANCE
            and len(cat_pois) >= 10
            and len(self.categories.get(category, [])) > 3
        ):
            types = [p["poi_type"] for p in cat_pois if p.get("poi_type")]
            if types:
                max_share = max(Counter(types).values()) / len(types)
                if max_share > 0.5:
                    # Linear penalty: 50% share → ×1.0, 100% share → ×0.8
                    raw_score *= 1.0 - (max_share - 0.5) * _DOMINANCE_PENALTY_MULTIPLIER

        return {
            "score":      round(float(np.clip(raw_score, 0, 100)), 2),
            "components": {k: round(float(v), 2) for k, v in components.items()},
        }

    # ------------------------------------------------------------------
    # Component 1 — Density (25%)
    # ------------------------------------------------------------------

    def _density(self, cat_pois: List[Dict], category: str) -> float:
        """
        Score based on POI count relative to India-calibrated urban benchmarks.

        DENSITY_THRESHOLDS stores a single float per category representing the
        "good" benchmark (POIs per km² in a typical urban area). We derive
        excellent/fair from it and use a log-saturating curve.
        """
        # Multi-Radius Weighted Density Score
        # 500m (50%) -> Walkable / Immediate
        # 1km  (30%) -> Neighborhood / Short Drive
        # 2km  (20%) -> Catchment / Regional
        
        radii_weights = [(0.5, 0.5), (1.0, 0.3), (2.0, 0.2)]
        final_score = 0.0
        
        benchmark = self.density_thresholds.get(category, 2.0)  # POIs/km²

        for r_km, weight in radii_weights:
            # count POIs within this radius — defensive .get() for robustness
            count = sum(1 for p in cat_pois if p.get("distance_km", 9999) <= r_km)
            
            # area of this specific radius
            area_km2 = np.pi * r_km ** 2
            
            # benchmark for this specific area
            good      = max(1, benchmark * area_km2)
            excellent = good * 2.5
            fair      = good * 0.4
            
            # Score this radius (0-100)
            if count == 0:
                r_score = 0.0
            elif count >= excellent:
                r_score = 100.0
            elif count >= good:
                r_score = 70.0 + 30.0 * (count - good) / (excellent - good)
            elif count >= fair:
                r_score = 40.0 + 30.0 * (count - fair) / (good - fair)
            else:
                r_score = max(0.0, 40.0 * count / max(fair, 1))
            
            final_score += r_score * weight

        return final_score

    # ------------------------------------------------------------------
    # Component 2 — Proximity (20%)
    # ------------------------------------------------------------------

    def _proximity(self, cat_pois: List[Dict], category: str) -> float:
        """
        70/30 blend of nearest-POI score and average-distance score.

        Nearest-POI (70%): steep decay — dictates urgent / immediate accessibility.
          lambda=1.5 → 0.1km=86, 0.5km=47, 1.0km=22, 2.0km=5
        Average-distance (30%): uses only the 5 nearest POIs.
          Averaging all POIs within 2km biases the score toward the circle edge;
          restricting to the closest few better represents real user access.
        """
        if not cat_pois:
            return 0.0
        decay = CATEGORY_PROXIMITY_DECAY_RATES.get(category, 1.5)
        sorted_dists = sorted(p.get("distance_km", 9999) for p in cat_pois)
        nearest_km = sorted_dists[0]
        closest_5 = sorted_dists[:5]
        avg_km = sum(closest_5) / len(closest_5)
        s_min = 100.0 * np.exp(-decay * nearest_km)
        s_avg = 100.0 * np.exp(-(decay / 1.5) * avg_km)  # softer decay for average
        return float(0.70 * s_min + 0.30 * s_avg)

    # ------------------------------------------------------------------
    # Component 3 — Quality (20%)
    # ------------------------------------------------------------------

    # Premium and basic POI types per category (Centralized in config)
    _PREMIUM = PREMIUM_POIS
    _BASIC = BASIC_POIS

    def _quality(self, cat_pois: List[Dict], category: str) -> float:
        """Quality score based on premium-to-basic POI ratio.

        Requires a minimum of 3 matched POIs to produce a reliable ratio.
        With fewer, the score is scaled down to reflect insufficient data.
        """
        if not cat_pois:
            return 0.0

        premium_types = self._PREMIUM.get(category, [])
        basic_types   = self._BASIC.get(category, [])
        premium = sum(1 for p in cat_pois if p.get("poi_type") in premium_types)
        basic   = sum(1 for p in cat_pois if p.get("poi_type") in basic_types)

        if premium == 0 and basic == 0:
            return 0.0

        relevant_total = premium + basic
        if relevant_total == 0:
            return 0.0

        premium_ratio = safe_divide(premium, relevant_total)
        basic_ratio   = safe_divide(basic, relevant_total)
        raw = float(np.clip(premium_ratio * 100 * 1.5 + basic_ratio * 100 * 0.5, 0, 100))

        # Scale down when we have too few POIs for a reliable ratio.
        # 1 POI → 33%, 2 POIs → 66%, >=3 POIs → full score.
        # This prevents a single high-weight POI from dominating the category.
        if relevant_total < 3:
            raw *= relevant_total / 3.0

        return raw

    # ------------------------------------------------------------------
    # Component 4 — Accessibility (15%)
    # ------------------------------------------------------------------

    def _accessibility(self, cat_pois: List[Dict], category: str) -> float:
        """Gravity-model score: Σ weight / distance².

        Normalised to 100 so scores span the full 0–100 range:
          10 POIs at 0.5km weight=1  → Σ = 10/0.25 = 40  → score = 40
          10 POIs at 0.1km weight=1  → Σ = 10/0.01 = 1000 → score = 100 (capped)
          5  POIs at 0.3km weight=1.5 → Σ = 5*1.5/0.09 ≈ 83 → score = 83
        """
        if not cat_pois:
            return 0.0
        weights = self.poi_weights.get(category, {})
        score = sum(
            weights.get(p.get("poi_type", ""), 1.0) / max(p.get("distance_km", 9999), 0.1) ** 2
            for p in cat_pois
        )
        return float(np.clip(score / 100.0 * 100, 0, 100))

    # ------------------------------------------------------------------
    # Component 5 — Spatial (10%)
    # ------------------------------------------------------------------

    def _spatial(self, cat_pois: List[Dict], category: str, features: Dict) -> float:
        """
        Spatial distribution quality.

        Returns 0 when fewer than 3 POIs — 1 or 2 isolated POIs give no
        meaningful spatial signal and should not contribute to the score.
        """
        if not cat_pois:
            return 0.0
        if len(cat_pois) < 3:
            return 0.0

        try:
            # Use category-specific NNI where available, fall back to global NNI.
            # Global NNI is used as fallback since per-category NNI is only computed
            # for categories with 3+ POIs (see _advanced_spatial_features).
            global_nni = features.get("global_nearest_neighbor_index", 1.0)
            nni = features.get(f"{category}_nearest_neighbor_index", global_nni)

            # Piecewise NNI score:
            # Optimal walkable range 0.5–1.0 scores 100.
            # Score drops for over-clustering (NNI < 0.5) or sprawl (NNI > 1.0).
            if nni < 0.5:
                nni_score = 50.0 + (nni / 0.5) * 50.0      # 0→50, 0.5→100
            elif nni <= 1.0:
                nni_score = 100.0                            # optimal range
            elif nni <= 1.5:
                nni_score = 100.0 - ((nni - 1.0) / 0.5) * 50.0  # 1.0→100, 1.5→50
            else:
                nni_score = max(0.0, 50.0 - (nni - 1.5) * 20.0)  # >1.5 → sprawl
            nni_score = float(np.clip(nni_score, 0, 100))

            # Sub-component B: Hotspot intensity (Category specific)
            hotspot = features.get(f"{category}_hotspot_intensity", 0.0)
            hotspot_score = float(np.clip(hotspot, 0, 100))

            # 70% Hotspot (Local signal) + 30% NNI (Global signal)
            return 0.7 * hotspot_score + 0.3 * nni_score

        except Exception as exc:
            logger.debug(f"Spatial component fallback for {category}: {exc}")
            return 0.0

    # ------------------------------------------------------------------
    # Component 6 — Economic (10%)
    # ------------------------------------------------------------------

    def _economic(self, cat_pois: List[Dict], all_pois: List[Dict], category: str) -> float:
        """
        Category share of total POIs vs India-calibrated target.

        Sigmoid centred at ratio=1.0 (at target) → 50 score.
          ratio = 0.5 → ~18   (below target)
          ratio = 1.0 → ~50   (at target)
          ratio = 2.0 → ~82   (double target)

        Minimum 3 POIs required for a stable economic signal. Fewer POIs
        are penalised because a single POI can artificially hit any target%.
        """
        if not cat_pois or not all_pois:
            return 0.0

        # Low confidence guard: 1-2 POIs cannot give a reliable economic signal
        n = len(cat_pois)
        if n < 3:
            confidence = n / 3.0  # 1 POI → 33%, 2 POIs → 66%
        else:
            confidence = 1.0

        category_pct = safe_divide(len(cat_pois) * 100, len(all_pois))
        target = ECONOMIC_TARGET_PCT.get(category, 10)
        ratio  = safe_divide(category_pct, target)

        raw = float(np.clip(100 / (1 + np.exp(-4.0 * (ratio - 1.0))), 0, 100))
        return raw * confidence

## amenity_calculator.py

In [ ]:
"""
amenity_calculator.py — Compute the final Amenity Index (0–100).

Aggregates weighted category scores and applies four additive penalties:
  1. Data quality  — penalises sparse OSM coverage
  2. Type Gini     — penalises unequal category distribution
  3. Diversity     — penalises low Simpson's diversity
  4. Missing essentials — penalises absence of critical categories

Classification thresholds (India-calibrated):
  Metro  ≥ 60
  Urban  ≥ 30
  Rural  < 30
"""

import logging
from typing import Dict, Optional

import numpy as np


logger = logging.getLogger(__name__)

# Minimum score a category must achieve to be considered "present"
_ESSENTIAL_THRESHOLD = 10.0
_ESSENTIAL_CATEGORIES = frozenset({"healthcare", "essential", "transport"})


class AmenityCalculator:
    """
    Compute the final Amenity Index from per-category scores.

    Usage:
        calc = AmenityCalculator()
        result = calc.calculate(category_scores, total_pois=120, features=features)
    """

    def calculate(
        self,
        category_scores: Dict[str, Dict],
        total_pois: int = 0,
        features: Optional[Dict] = None,
    ) -> Dict:
        """
        Compute Amenity Index and classification.

        Args:
            category_scores: {category: {"score": float, "components": {...}}}
            total_pois:      Total POI count (for data quality penalty).
            features:        Feature dict (for Gini / Simpson penalties).

        Returns:
            {
                "amenity_index":  float,   # 0–100
                "classification": str,     # Metro / Urban / Rural
                "data_quality":   str,     # High / Medium / Low / Zero
                "penalties":      dict,    # breakdown of applied penalties
                "weighted_score": float,   # pre-penalty score
            }
        """
        features = features or {}

        # ── Zero-POI guard ───────────────────────────────────────────
        # If no POIs were fetched (API failure / truly empty area),
        # return 0 immediately rather than a spurious floor score.
        if total_pois == 0:
            return {
                "amenity_index":  0.0,
                "classification": "Rural",
                "data_quality":   "Zero",
                "penalties":      {},
                "weighted_score": 0.0,
            }

        # ── Weighted base score ──────────────────────────────────────
        weighted = sum(
            category_scores.get(cat, {}).get("score", 0.0) * weight
            for cat, weight in CATEGORY_WEIGHTS.items()
        )

        # ── Penalties (additive, capped at 50% total) ────────────────
        penalties: Dict[str, float] = {}

        # 1. Data quality
        thresholds = DATA_QUALITY_POI_THRESHOLDS
        pen_map    = DATA_QUALITY_PENALTIES
        if total_pois < thresholds["very_sparse"]:
            penalties["data_quality"] = pen_map["very_sparse"]
        elif total_pois < thresholds["sparse"]:
            penalties["data_quality"] = pen_map["sparse"]
        elif total_pois < thresholds["moderate"]:
            penalties["data_quality"] = pen_map["moderate"]
        else:
            penalties["data_quality"] = 0.0

        # 2. Type Gini — continuous linear penalty: P = G × 0.15
        # Gini 0 = perfect equality → no penalty; Gini 1 = all POIs one category → 15% penalty
        gini = features.get("global_gini_coefficient", 0.0)
        penalties["type_gini"] = float(np.clip(gini * 0.15, 0.0, 0.15))

        # 3. Simpson's diversity — continuous linear penalty: P = (1−D)×0.10
        # simpson is on 0–100 scale; (1 − D/100) × 0.10 gives 0–10% penalty
        simpson = features.get("global_simpson_diversity", 100.0)
        penalties["diversity"] = float(np.clip((1 - simpson / 100.0) * 0.10, 0.0, 0.10))

        # 4. Missing essential categories: 3%/missing, cap 9%
        missing = sum(
            1 for cat in _ESSENTIAL_CATEGORIES
            if category_scores.get(cat, {}).get("score", 0.0) < _ESSENTIAL_THRESHOLD
        )
        penalties["missing_essentials"] = min(missing * 0.03, 0.09)

        # Cap total penalty at 50%
        total_penalty = min(sum(penalties.values()), 0.50)
        amenity_index = float(np.clip(weighted * (1 - total_penalty), 0, 100))

        return {
            "amenity_index":  round(amenity_index, 2),
            "classification": self._classify(amenity_index),
            "data_quality":   self._data_quality_label(total_pois),
            "penalties":      {k: round(v, 4) for k, v in penalties.items()},
            "weighted_score": round(weighted, 2),
        }

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _classify(score: float) -> str:
        if score >= 60:
            return "Metro"
        if score >= 30:
            return "Urban"
        return "Rural"

    @staticmethod
    def _data_quality_label(total_pois: int) -> str:
        """Labels aligned with DATA_QUALITY_POI_THRESHOLDS (5, 20, 40)."""
        if total_pois == 0:
            return "Zero"
        if total_pois < 5:
            return "Very Low"
        if total_pois < 20:
            return "Low"
        if total_pois < 40:
            return "Medium"
        return "High"

    # Backwards-compatible alias used by existing main.py
    def calculate_amenity_index(self, category_scores, total_pois=0, features=None):
        return self.calculate(category_scores, total_pois=total_pois, features=features)



## Run the Pipeline
Test it on a single location!

In [ ]:
# You can change these coordinates to test any location in India
location = {
    'latitude': 18.9220,
    'longitude': 72.8347,
    'address': 'Kala Ghoda, Mumbai'
}
print(f"Target Location: {location['address']}")

fetcher = POIFetcher()
extractor = FeatureExtractor()
scorer = CategoryScorer()
calculator = AmenityCalculator()

print("Fetching POIs from OpenStreetMap...")
pois = fetcher.fetch(location['latitude'], location['longitude'])
print(f"Found {len(pois)} POIs within a 2km radius.")

print("Extracting spatial and density features...")
features = extractor.extract_all(location['latitude'], location['longitude'], pois)

print("Scoring categories...")
cat_scores = {cat: scorer.score(cat, features, pois) for cat in CATEGORIES}
final = calculator.calculate_amenity_index(cat_scores, total_pois=len(pois), features=features)

print("\n" + "="*50)
print(" 🏆 FINAL RESULT")
print("="*50)
print(f"Amenity Index :  {final['amenity_index']} / 100")
print(f"Classification:  {final['classification']}")
print(f"Data Quality  :  {final['data_quality']}")
print("-"*50)
print("Penalties Applied:")
for p, val in final['penalties'].items():
    print(f"  {p:<20}: {val*100:.2f}%")
print("="*50)
